In [1]:
import pickle
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import codebase

In [3]:
huc_dir = '/global/home/users/ann_scheliga/neuralhydrology/data/CAMELS_US/shapefiles/'

In [4]:
huc_filelist = os.listdir(huc_dir)

r_fn = re.compile("Region_[0-9]{2}_nhru_simplify_100.shp$") # grab all unzipped huc boundary shp files
huc_shp_files = list(filter(r_fn.match,huc_filelist))

In [5]:
huc_shp_dict = {} # store huc geodataframes with Region number as the key
to_skip = [] # regionIDs (as numbers) to not load
for filename in huc_shp_files:
    huc_num = int(re.findall(r'Region_([0-9]{2})_', filename)[0]) # grab the Region number. Is an int , not str
    if huc_num in to_skip: pass
    else:
        huc_shp_dict[huc_num] = gpd.read_file(Path(huc_dir, filename))
huc_shp_dict.keys()

ERROR 1: PROJ: proj_create_from_database: Open of /global/home/users/ann_scheliga/.conda/envs/rio_keras/share/proj failed


dict_keys([15, 3])

In [6]:
print(huc_shp_dict[3].columns)
huc_shp_dict[3].head()

Index(['Dissolve', 'POI_ID', 'PROD_UNIT', 'hru_id_pro', 'hru_id_reg',
       'hru_id_loc', 'hru_id', 'hru_x', 'hru_y', 'hru_area', 'hru_elev',
       'hru_slope', 'hru_aspect', 'asp_sin', 'asp_cos', 'jh_coef_hr',
       'tmin_adj', 'tmax_adj', 'snarea_thr', 'hru_deplcr', 'soil_moist',
       'soil_rechr', 'soil_type', 'cov_type', 'VALUE_0', 'VALUE_1', 'VALUE_2',
       'VALUE_3', 'covden_sum', 'leaf_loss', 'covden_win', 'rad_trncf',
       'srain_intc', 'wrain_intc', 'snow_intcp', 'hru_percen', 'soil2gw_ma',
       'ssr2gw_rat', 'fastcoef_l', 'slowcoef_l', 'gwflow_coe', 'dprst_seep',
       'dprst_se_1', 'dprst_flow', 'r_k_perm_w', 'r_junk', 'hru_long',
       'hru_lat', 'dprst_area', 'sro_to_dpr', 'hru_segmen', 'Shape_Leng',
       'Shape_Area', 'hru_segm_1', 'GAGEID', 'geometry'],
      dtype='object')


,Dissolve,POI_ID,PROD_UNIT,hru_id_pro,hru_id_reg,hru_id_loc,hru_id,hru_x,hru_y,hru_area,...,hru_long,hru_lat,dprst_area,sro_to_dpr,hru_segmen,Shape_Leng,Shape_Area,hru_segm_1,GAGEID,geometry
0,2593,8719481,3e,1354,9222,1,9222,1.603502e+06,1.712429e+06,16872.893610,...,-77.683896,37.059507,22.078535,0.175384,2,60317.082869,6.828218e+07,1581,02046000,"POLYGON ((1609369.173 1716576.947, 1610073.405..."
1,2611,8719481,3e,1369,9237,2,9237,1.599911e+06,1.718527e+06,33165.301445,...,-77.710672,37.119064,164.117561,0.621347,2,109889.533875,1.342152e+08,1581,02046000,"POLYGON ((1601024.980 1724084.975, 1601265.247..."
2,2657,8718687,3e,1413,9281,3,9281,1.591499e+06,1.713158e+06,21840.478348,...,-77.816095,37.086343,40.017678,0.078187,1,81361.730219,8.838528e+07,1579,02046000,"MULTIPOLYGON (((1590855.311 1716525.130, 15908..."
3,2659,8744593,3e,1415,9283,1,9283,1.543893e+06,1.699976e+06,35856.492300,...,-78.375046,37.050352,14.372039,0.058901,1,90359.301849,1.451061e+08,1636,02051000,"POLYGON ((1547054.668 1706595.143, 1548075.193..."
4,2229,8741021,3e,1127,8995,1,8995,1.592421e+06,1.666849e+06,17324.295472,...,-77.905347,36.678971,0.000000,0.000000,2,64592.981185,7.010894e+07,1614,02051500,"MULTIPOLYGON (((1598483.785 1672522.496, 15984..."


In [7]:
huc_shp_dict[3].explore(tooltip='GAGEID')

In [8]:
huc3 = huc_shp_dict[3]
gageID = '02235200'
test_HUC = huc3[huc3['GAGEID'] == gageID]

In [9]:
print(os.getenv("LD_LIBRARY_PATH"))

None


In [10]:
test_HUC_latlon = test_HUC.to_crs(epsg=4326)

In [11]:
test_HUC_latlon.geometry.buffer(0).bounds.loc[209].to_dict()

{'minx': -81.68618774452528,
 'miny': 28.84624862701476,
 'maxx': -81.42883300742223,
 'maxy': 29.054162979154427}

In [12]:
## Read in result


In [2]:
pd.read_csv('/global/scratch/users/ann_scheliga/CYGNSS_daily/time_series/test_02235200.csv')

,Unnamed: 0,Filename,area
0,2019-01-01,cyg.ddmi.2019-01-01.l3.uc-berkeley-watermask-d...,5.895303e+07
1,2019-01-02,cyg.ddmi.2019-01-02.l3.uc-berkeley-watermask-d...,5.790029e+07
2,2019-01-03,cyg.ddmi.2019-01-03.l3.uc-berkeley-watermask-d...,6.842762e+07
3,2019-01-04,cyg.ddmi.2019-01-04.l3.uc-berkeley-watermask-d...,7.053309e+07
4,2019-01-05,cyg.ddmi.2019-01-05.l3.uc-berkeley-watermask-d...,6.526942e+07
...,...,...,...
1822,2023-12-28,cyg.ddmi.2023-12-28.l3.uc-berkeley-watermask-d...,5.684756e+07
1823,2023-12-29,cyg.ddmi.2023-12-29.l3.uc-berkeley-watermask-d...,4.842570e+07
1824,2023-12-30,cyg.ddmi.2023-12-30.l3.uc-berkeley-watermask-d...,5.158390e+07
1825,2023-12-31,cyg.ddmi.2023-12-31.l3.uc-berkeley-watermask-d...,5.053117e+07
